PePy

Author: Julia K. Varga <jvarga92@gmail.com>  
License: BSD 3 clause  
Code Repository: https://github.com/gezmi/pepy

In [ ]:
import pandas as pd
pd.set_option('display.width', 600)
pd.set_option('display.max_columns', 8)

# Working with Confidence Data

PePy can load confidence metrics produced by structure prediction methods — AlphaFold2/ColabFold, AlphaFold3, and ChAI.

Confidence data includes:
- **PAE matrix** — Predicted Aligned Error between all residue pairs
- **iPTM** — interface predicted TM-score
- **pTM** — predicted TM-score
- **pLDDT** — per-residue confidence (stored in B-factor column of structure files)

This tutorial shows how to load and use confidence data from each format.

In [ ]:
from pepy import ProteinComplex

## AlphaFold2 / ColabFold

AF2 stores confidence data in a JSON file alongside the PDB structure. PePy auto-discovers it by converting the structure filename:

- `*_unrelaxed_*` → `*_scores_*`
- `.pdb` → `.json`

In [ ]:
af2 = ProteinComplex.from_file(
    '../../pepy/tests/data/1ycr_af2_55d19_unrelaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000.pdb'
)
af2.identify_chains()

# Load confidence — auto-discovers the JSON
loaded = af2.load_confidence_data()
print(f'Confidence loaded: {loaded}')
af2.get_confidence_summary()

## AlphaFold3

AF3 uses two JSON files:
- `*_full_data_*.json` — contains the PAE matrix
- `*_summary_confidences_*.json` — contains iPTM, pTM, etc.

Both are merged automatically:

In [ ]:
af3 = ProteinComplex.from_file('../../pepy/tests/data/fold_1ycr_af3_model_0.cif')
af3.identify_chains()

loaded = af3.load_confidence_data()
print(f'Confidence loaded: {loaded}')
af3.get_confidence_summary()

## ChAI

ChAI stores confidence data in a NumPy `.npz` file alongside the PDB structure.
PePy matches them by the model index in the filename:

In [ ]:
chai = ProteinComplex.from_file('../../pepy/tests/data/pred.model_idx_0.pdb')
chai.identify_chains()

loaded = chai.load_confidence_data()
print(f'Confidence loaded: {loaded}')
chai.get_confidence_summary()

## Explicit confidence file path

If auto-discovery doesn't find your file (e.g. non-standard naming), you can provide the path explicitly:

In [ ]:
af2.load_confidence_data(
    confidence_file_path='../../pepy/tests/data/1ycr_af2_55d19_scores_rank_001_alphafold2_multimer_v3_model_1_seed_000.json'
)
af2.get_confidence_summary()

## Accessing the Raw Data

After loading, the raw confidence data is available on the complex object:

In [ ]:
print(f"iPTM: {af2.confidence_data['iptm']}")
print(f"pTM:  {af2.confidence_data['ptm']}")
print(f"Combined (0.8*iPTM + 0.2*pTM): {af2.confidence_data['confidence']}")

### PAE matrix

The PAE matrix is a pandas DataFrame — rows and columns correspond to CA atom indices in the structure:

In [ ]:
pae = af2.confidence_data['pae_matrix']
print(f'PAE matrix shape: {pae.shape}')
pae.iloc[:5, :5]

### pLDDT values

Per-residue pLDDT is stored in the B-factor column of the structure, not in the confidence file:

In [ ]:
ca_atoms = af2.df['ATOM'][af2.df['ATOM']['atom_name'] == 'CA']
ca_atoms[['chain_id', 'residue_number', 'residue_name', 'b_factor']].head(10)

## Interface Confidence Metrics

`calculate_confidence_metrics()` combines interface residues with the confidence data to compute interface-specific metrics.

This requires both the interface and confidence data to be loaded first:

In [ ]:
# Calculate interface first
af2.calculate_interface()

# Then compute confidence metrics for the interface
metrics = af2.calculate_confidence_metrics()
metrics

Key metrics:

| Metric | Description |
|--------|-------------|
| `avg_plddt_interface` | Mean pLDDT of binder interface CA atoms |
| `max_plddt_interface` | Max pLDDT of binder interface CA atoms |
| `interface_pae` | Median PAE between binder–receptor interface residues |
| `min_interface_pae` | Min PAE between binder–receptor interface residues |
| `iptm` | Interface predicted TM-score (global) |
| `ptm` | Predicted TM-score (global) |
| `combined_confidence` | 0.8 × iPTM + 0.2 × pTM |

### Interface PAE submatrix

You can also extract the PAE submatrix for just the interface residues. This uses the `ca_index` column that maps each residue to its position in the PAE matrix:

In [ ]:
atom_df = af2.df['ATOM']
pae = af2.confidence_data['pae_matrix']

# Get CA indices for interface residues
binder_ca = atom_df[
    (atom_df['chain_id'].isin(af2.binder_chains)) &
    (atom_df['residue_number'].isin(af2.interface_residues_binder)) &
    (atom_df['atom_name'] == 'CA')
]
receptor_ca = atom_df[
    (atom_df['chain_id'].isin(af2.receptor_chains)) &
    (atom_df['residue_number'].isin(af2.interface_residues_receptor)) &
    (atom_df['atom_name'] == 'CA')
]

# Extract submatrix
binder_idx = binder_ca['ca_index'].tolist()
receptor_idx = receptor_ca['ca_index'].tolist()
pae_sub = pae.iloc[binder_idx, receptor_idx]

print(f'Interface PAE submatrix: {pae_sub.shape}')
print(f'Median: {pae_sub.median().median():.2f}')
print(f'Min:    {pae_sub.min().min():.2f}')
pae_sub

## Adding Support for New Formats

The confidence file discovery uses pattern lists defined at the top of `pepy/io/confidence.py`. To add support for a new prediction method:

1. Add filename patterns to `AF2_STRIP_PATTERNS` / `AF2_REPLACEMENTS` (for PDB-based outputs)
2. Or add new glob patterns in `_find_af3_or_chai()` (for CIF/NPZ-based outputs)
3. If the file uses new metric key names, add them to `METRIC_KEYS`

In [ ]:
from pepy.io.confidence import AF2_STRIP_PATTERNS, AF2_REPLACEMENTS, METRIC_KEYS

print('Strip patterns:', AF2_STRIP_PATTERNS)
print('Replacements:', AF2_REPLACEMENTS)
print('Metric keys:', METRIC_KEYS)